# 03 — Sağlamlık paketi (E6, GPU)
**Çalışma zamanı → GPU.** Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`.
**Kullanım:** ilk kod hücresini kontrol et, sonra **Çalışma zamanı → Tümünü çalıştır**.
Her bozulma koşulu için val'in bozulmuş kopyası üretilir, uygun her model değerlendirilir,
kopya silinir. Oturum koparsa yeniden **Tümünü çalıştır**: biten (model, koşul) çiftleri atlanır.
Çıktılar: `MyDrive/baykus/results/robust/*.json`, tahminler `MyDrive/baykus/robust/`.

In [ ]:
CONFIGS = ['configs/e1_rgb.yaml', 'configs/e2_ir.yaml', 'configs/e3_early_fusion.yaml']
BRANCH = 'main'                  # haftalık dalı denemek için ör. 'h3'
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
!uv pip install --system -q -e '/content/baykus/python[train]'
import sys
sys.path.insert(0, '/content/baykus/python/src')

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
# Modeller: her config'in biten en son tam eğitiminin best.pt'si
import os, yaml
from baykus.train.checkpoints import find_runs
MODEL_ARGS = []
for c in CONFIGS:
    exp = yaml.safe_load(open(f'/content/baykus/{c}', encoding='utf-8'))['experiment']
    done = find_runs(f'{DRIVE}/runs', exp, finished=True)
    assert done, f'{exp}: biten tam eğitim yok'
    MODEL_ARGS.append(f'--model ../{c}={done[-1]}/weights/best.pt')
    print(exp, '→', done[-1])
NPROC = os.cpu_count()
!du -sh {DRIVE}/eval {DRIVE}/robust 2>/dev/null   # Drive'da yer: tahmin dosyaları büyük olabilir

In [ ]:
%%time
# E6: sağlamlık paketi (val, IR etiketleri). Süreyi ve son 20 satırı Claude'a getir.
ARGS = ' '.join(MODEL_ARGS)
!cd /content/baykus/python && baykus-robust run {ARGS} --data-root /content/dv_rgbt --meta /content/meta --work /content/robust_work --project {DRIVE}/robust --results {DRIVE}/results --workers {NPROC}